# Corrected FDA Consensus Raman Spectrum

This Colab-ready notebook implements the corrected real-data consensus workflow. It uses the structured tabular datasets already in the repository, applies Savitzky-Golay smoothing, ALS baseline correction, negative clipping, L2 normalization, and then fits cubic B-spline functional representations for consensus construction.

## 1. Setup

In [ ]:
# Colab/local setup. Installs only missing runtime packages.
import importlib
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/arjun041008/RamanSpectroscopy.git"
REPO_DIRNAME = "RamanSpectroscopy"

def ensure_package(import_name, package_name=None):
    package_name = package_name or import_name
    try:
        return importlib.import_module(import_name)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package_name])
        return importlib.import_module(import_name)

for import_name, package_name in [
    ("numpy", "numpy"),
    ("pandas", "pandas"),
    ("matplotlib", "matplotlib"),
    ("scipy", "scipy"),
    ("sklearn", "scikit-learn"),
    ("pywt", "PyWavelets"),
    ("openpyxl", "openpyxl"),
]:
    ensure_package(import_name, package_name)

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    base = Path("/content")
    repo_root = base / REPO_DIRNAME
    if not repo_root.exists():
        subprocess.check_call(["git", "clone", REPO_URL, str(repo_root)])
    os.chdir(repo_root)
else:
    current = Path.cwd().resolve()
    candidates = [current, *current.parents]
    repo_root = next((p for p in candidates if (p / "src" / "combined_raman_spectra.xlsx").exists()), current)
    os.chdir(repo_root)

print(f"Repository root: {Path.cwd()}")

In [ ]:
# Materialize the corrected shared pipeline inside the Colab-cloned repo.
# This makes the notebook shareable even before the corrected branch is pushed.
from pathlib import Path

HELPER_SOURCE = 'from __future__ import annotations\n\nimport json\nimport math\nimport os\nimport re\nimport warnings\nfrom dataclasses import dataclass\nfrom pathlib import Path\n\nimport matplotlib.pyplot as plt\nimport numpy as np\nimport pandas as pd\nimport pywt\nfrom scipy import sparse\nfrom scipy.interpolate import interp1d, splrep, splev\nfrom scipy.signal import find_peaks, savgol_filter\nfrom scipy.sparse.linalg import spsolve\nfrom scipy.stats import wasserstein_distance\nfrom sklearn.decomposition import NMF\nfrom sklearn.metrics import auc, roc_auc_score, roc_curve\n\n\nREPO_ROOT = Path(__file__).resolve().parents[1]\nSRC_DIR = REPO_ROOT / "src"\nMILK_DIR = SRC_DIR / "Milk_AI"\nOUT_ROOT = REPO_ROOT / "validation_output"\nCORRECTED_DIR = OUT_ROOT / "corrected_pipeline"\nNOTEBOOK3_DIR = OUT_ROOT / "notebook3"\nDOWNLOADS = Path(os.environ.get("RAW_SAMPLE_DIR", r"C:\\Users\\Amay Kashyap Deka\\Downloads"))\n\nCONTROL_XLSX = SRC_DIR / "combined_raman_spectra.xlsx"\nEXPERIMENTAL_XLSX = SRC_DIR / "combined_raman_spectra (Experimental).xlsx"\n\nCOMMON_SHIFT = np.arange(400.0, 3500.0 + 0.625, 1.25)\nFINGERPRINT_BOUNDS = (400.0, 1800.0)\nCH_STRETCH_BOUNDS = (2800.0, 3000.0)\nTHRESHOLD_PERCENTILE = 95\nRANDOM_STATE = 42\n\n\nLITERATURE_TABLE = pd.DataFrame(\n    [\n        (1016, "Aromatic C-C", "Fat", "milk"),\n        (1279, "CH2 twist / ester", "Fat", "milk"),\n        (1315, "CH2 twist / ester", "Fat", "milk"),\n        (1416, "CH2 bending", "Fat / Lactose", "milk"),\n        (1759, "CH2 bending", "Fat / Lactose", "milk"),\n        (1566, "Amide I", "Proteins", "milk"),\n        (1670, "Amide II", "Proteins", "milk"),\n        (2865, "CH stretch", "Fatty acids", "milk"),\n        (2902, "CH stretch", "Fatty acids", "milk"),\n        (1160, "Carotenoids", "Bacterial-action signature", "bacterial-action"),\n        (700, "Nucleic-acid region", "Bacterial-action signature", "bacterial-action"),\n        (800, "Nucleic-acid region", "Bacterial-action signature", "bacterial-action"),\n        (1450, "Acetyl-CoA / beta-oxidation", "Bacterial-action signature", "bacterial-action"),\n        (724, "Acetyl-CoA / beta-oxidation", "Bacterial-action signature", "bacterial-action"),\n    ],\n    columns=["band_cm-1", "functional_group", "assignment", "source_type"],\n)\n\n\n@dataclass\nclass Spectrum:\n    name: str\n    group: str\n    x: np.ndarray\n    y: np.ndarray\n    source: str\n\n\ndef ensure_dirs() -> None:\n    CORRECTED_DIR.mkdir(parents=True, exist_ok=True)\n    NOTEBOOK3_DIR.mkdir(parents=True, exist_ok=True)\n\n\ndef als_baseline(y: np.ndarray, lam: float = 1e6, p: float = 0.01, n_iter: int = 10) -> np.ndarray:\n    """Asymmetric least-squares baseline used by the original preprocessing notebooks."""\n    y = np.asarray(y, dtype=float)\n    length = y.size\n    difference = sparse.diags(\n        [np.ones(length), -2.0 * np.ones(length), np.ones(length)],\n        [0, -1, -2],\n        shape=(length, length - 2),\n        format="csc",\n    )\n    weights = np.ones(length)\n    for _ in range(n_iter):\n        weight_matrix = sparse.spdiags(weights, 0, length, length)\n        system = weight_matrix + lam * difference.dot(difference.T)\n        baseline = spsolve(system, weights * y)\n        weights = p * (y > baseline) + (1.0 - p) * (y < baseline)\n    return baseline\n\n\ndef preprocess_native(x: np.ndarray, y: np.ndarray) -> dict[str, np.ndarray]:\n    """Savitzky-Golay -> ALS baseline correction -> negative clipping -> L2 normalization."""\n    y = np.asarray(y, dtype=float)\n    smoothed = savgol_filter(y, window_length=5, polyorder=2)\n    baseline = als_baseline(smoothed, lam=1e6, p=0.01, n_iter=10)\n    corrected = smoothed - baseline\n    clipped = np.clip(corrected, 0.0, None)\n    norm = np.linalg.norm(clipped)\n    if not np.isfinite(norm) or norm <= 0:\n        raise ValueError("Preprocessing produced a zero/non-finite spectrum.")\n    normalized = clipped / norm\n    return {\n        "raw": y,\n        "smoothed": smoothed,\n        "baseline": baseline,\n        "baseline_corrected": clipped,\n        "normalized": normalized,\n    }\n\n\ndef fit_functional_spline(x: np.ndarray, y: np.ndarray):\n    """Fit a common-order B-spline functional representation.\n\n    s=1e-4 is applied to every normalized spectrum. This is deliberately small:\n    the prior Savitzky-Golay + ALS pipeline already removes most high-frequency\n    acquisition noise, so the spline primarily provides a continuous FDA object\n    and consistent evaluation on a common Raman grid rather than aggressive\n    smoothing.\n    """\n    return splrep(x, y, k=3, s=1e-4)\n\n\ndef spline_eval(tck, grid: np.ndarray = COMMON_SHIFT) -> np.ndarray:\n    return np.clip(np.asarray(splev(grid, tck), dtype=float), 0.0, None)\n\n\ndef linear_eval(x: np.ndarray, y: np.ndarray, grid: np.ndarray = COMMON_SHIFT) -> np.ndarray:\n    f = interp1d(x, y, kind="linear", bounds_error=False, fill_value="extrapolate", assume_sorted=True)\n    return np.clip(f(grid), 0.0, None)\n\n\ndef geometric_median(points: np.ndarray, tolerance: float = 1e-6, max_iterations: int = 500) -> np.ndarray:\n    estimate = np.mean(points, axis=0)\n    for _ in range(max_iterations):\n        distances = np.linalg.norm(points - estimate, axis=1)\n        if np.any(distances < 1e-12):\n            return points[np.argmin(distances)].copy()\n        inv = 1.0 / distances\n        updated = np.average(points, axis=0, weights=inv)\n        if np.linalg.norm(updated - estimate) < tolerance:\n            return updated\n        estimate = updated\n    return estimate\n\n\ndef exact_quantile_barycenter(spectra: np.ndarray, support: np.ndarray = COMMON_SHIFT, n_quantiles: int = 10001) -> np.ndarray:\n    nonnegative = np.clip(spectra, 0.0, None)\n    probabilities = nonnegative / nonnegative.sum(axis=1, keepdims=True)\n    levels = np.linspace(0.0, 1.0, n_quantiles)\n    quantiles = []\n    for row in probabilities:\n        positive = row > 0\n        xp = support[positive]\n        cp = np.cumsum(row[positive])\n        cp /= cp[-1]\n        quantiles.append(np.interp(levels, np.r_[0.0, cp], np.r_[xp[0], xp]))\n    mean_q = np.mean(quantiles, axis=0)\n    cdf = np.interp(support, mean_q, levels, left=0.0, right=1.0)\n    density = np.clip(np.gradient(cdf, support), 0.0, None)\n    density = np.clip(savgol_filter(density, 5, 2), 0.0, None)\n    density /= density.sum()\n    target_auc = np.mean(np.trapezoid(nonnegative, support, axis=1))\n    density *= target_auc / np.trapezoid(density, support)\n    return density\n\n\ndef sinkhorn_barycenter_numpy(spectra: np.ndarray, support: np.ndarray, reg: float, max_iter: int = 250) -> tuple[np.ndarray | None, dict]:\n    """Small NumPy Sinkhorn barycenter trial used only to audit low-reg behavior."""\n    try:\n        nonnegative = np.clip(spectra, 0.0, None)\n        probability = nonnegative / nonnegative.sum(axis=1, keepdims=True)\n        A = probability.T\n        x = ((support - support.min()) / np.ptp(support))[:, None]\n        M = (x - x.T) ** 2\n        M /= M.max()\n        K = np.exp(-M / reg)\n        if not np.isfinite(K).all() or np.count_nonzero(K) == 0:\n            return None, {"reg": reg, "stable": False, "reason": "kernel underflow/non-finite"}\n        UKv = K.dot((A.T / K.sum(axis=0)).T)\n        u = (np.exp(np.mean(np.log(np.maximum(UKv, 1e-300)), axis=1)) / np.maximum(UKv.T, 1e-300)).T\n        err = math.inf\n        for iteration in range(max_iter):\n            Ku = np.maximum(K.dot(u), 1e-300)\n            UKv = u * K.T.dot(A / Ku)\n            log_ukv = np.log(np.maximum(UKv, 1e-300))\n            geo = np.exp(np.mean(log_ukv, axis=1))\n            u = (u.T * geo).T / np.maximum(UKv, 1e-300)\n            if iteration % 10 == 1:\n                err = float(np.std(UKv, axis=1).sum())\n                if not np.isfinite(err):\n                    return None, {"reg": reg, "stable": False, "reason": "non-finite error", "iteration": iteration}\n                if err < 1e-6:\n                    break\n        bary = np.exp(np.mean(np.log(np.maximum(UKv, 1e-300)), axis=1))\n        bary = np.clip(bary, 0.0, None)\n        if not np.isfinite(bary).all() or bary.sum() <= 0:\n            return None, {"reg": reg, "stable": False, "reason": "non-finite/zero barycenter"}\n        target_auc = np.mean(np.trapezoid(nonnegative, support, axis=1))\n        bary *= target_auc / np.trapezoid(bary, support)\n        return bary, {\n            "reg": reg,\n            "stable": True,\n            "converged": err < 1e-6,\n            "final_error": err,\n            "iterations": iteration + 1,\n            "peak_to_median": float(bary.max() / np.median(bary[bary > 0])),\n        }\n    except Exception as exc:\n        return None, {"reg": reg, "stable": False, "reason": repr(exc)}\n\n\ndef load_tabular_controls() -> list[Spectrum]:\n    main = pd.read_excel(CONTROL_XLSX)\n    experimental = pd.read_excel(EXPERIMENTAL_XLSX)\n    spectra: list[Spectrum] = []\n    x_main = main["Raman Shift"].to_numpy(float)\n    for col in main.columns:\n        if col == "Raman Shift":\n            continue\n        spectra.append(Spectrum(col, "control", x_main, main[col].to_numpy(float), str(CONTROL_XLSX.relative_to(REPO_ROOT))))\n    x_exp = experimental["Raman Shift"].to_numpy(float)\n    non_bacteria = [c for c in experimental.columns if "NON BACTERIA" in c.upper()]\n    for col in non_bacteria:\n        spectra.append(Spectrum(col, "control", x_exp, experimental[col].to_numpy(float), str(EXPERIMENTAL_XLSX.relative_to(REPO_ROOT))))\n    return spectra\n\n\ndef load_bacterial_tabular() -> list[Spectrum]:\n    experimental = pd.read_excel(EXPERIMENTAL_XLSX)\n    x_exp = experimental["Raman Shift"].to_numpy(float)\n    spectra = []\n    for col in experimental.columns:\n        upper = col.upper()\n        if col == "Raman Shift":\n            continue\n        if "BACTERIA" in upper and "NON BACTERIA" not in upper:\n            spectra.append(Spectrum(col, "bacterial", x_exp, experimental[col].to_numpy(float), str(EXPERIMENTAL_XLSX.relative_to(REPO_ROOT))))\n    return spectra\n\n\ndef load_uploaded_raw_samples() -> tuple[list[Spectrum], list[str]]:\n    expected = [\n        "Sample 1 532nm 50% 600g 50xL 10s 10times 400-3500cm-1 200hole.txt",\n        "Sample 2 532nm 50% 600g 50xL 10s 10times 400-3500cm-1 200hole.txt",\n        "Sample 3 532nm 50% 600g 50xL 10s 10times 400-3500cm-1 200hole.txt",\n        "Sample 4 532nm 50% 600g 50xL 10s 10times 400-3500cm-1 200hole.txt",\n        "Sample 5 532nm 50% 600g 50xL 8s 10times 400-3500cm-1 200hole.txt",\n        "Sample 6 532nm 50% 600g 50xL 8s 10times 400-3500cm-1 200hole.txt",\n    ]\n    spectra = []\n    missing = []\n    for name in expected:\n        path = DOWNLOADS / name\n        if not path.exists():\n            missing.append(name)\n            continue\n        rows = []\n        skipped = 0\n        for line in path.read_text(encoding="utf-8", errors="replace").splitlines():\n            if not line.strip() or line.lstrip().startswith("#"):\n                skipped += 1\n                continue\n            parts = line.replace(",", " ").split()\n            if len(parts) < 2:\n                skipped += 1\n                continue\n            try:\n                rows.append((float(parts[0]), float(parts[1])))\n            except ValueError:\n                skipped += 1\n        data = np.asarray(rows, dtype=float)\n        spectra.append(Spectrum(name, "raw_unadulterated", data[:, 0], data[:, 1], str(path)))\n    return spectra, missing\n\n\ndef process_spectra(spectra: list[Spectrum]) -> tuple[pd.DataFrame, np.ndarray, dict[str, dict[str, np.ndarray]], np.ndarray]:\n    records = []\n    stages_by_name = {}\n    spline_matrix = []\n    linear_matrix = []\n    for spectrum in spectra:\n        stages = preprocess_native(spectrum.x, spectrum.y)\n        stages_by_name[spectrum.name] = stages\n        tck = fit_functional_spline(spectrum.x, stages["normalized"])\n        spline_y = spline_eval(tck)\n        linear_y = linear_eval(spectrum.x, stages["normalized"])\n        spline_matrix.append(spline_y)\n        linear_matrix.append(linear_y)\n        raw_baseline_ratio = float(np.percentile(spectrum.y, 95) / max(np.percentile(stages["baseline_corrected"], 95), 1e-12))\n        records.append(\n            {\n                "sample_id": spectrum.name,\n                "group": spectrum.group,\n                "source": spectrum.source,\n                "native_points": len(spectrum.x),\n                "x_min": float(spectrum.x.min()),\n                "x_max": float(spectrum.x.max()),\n                "median_spacing": float(np.median(np.diff(spectrum.x))),\n                "raw_min": float(spectrum.y.min()),\n                "raw_max": float(spectrum.y.max()),\n                "baseline_ratio_p95_raw_to_corrected": raw_baseline_ratio,\n            }\n        )\n    return pd.DataFrame(records), np.vstack(spline_matrix), stages_by_name, np.vstack(linear_matrix)\n\n\ndef plot_preprocessing_diagnostic(spectrum: Spectrum, stages: dict[str, np.ndarray], out_path: Path) -> None:\n    fig, axes = plt.subplots(4, 1, figsize=(13, 10), sharex=True)\n    axes[0].plot(spectrum.x, stages["raw"], lw=0.9)\n    axes[0].set_title("Raw tabular spectrum")\n    axes[1].plot(spectrum.x, stages["smoothed"], lw=0.9, color="#ff7f0e")\n    axes[1].set_title("Savitzky-Golay smoothed (window=5, polyorder=2)")\n    axes[2].plot(spectrum.x, stages["baseline_corrected"], lw=0.9, color="#2ca02c")\n    axes[2].set_title("ALS baseline-corrected and negative-clipped")\n    axes[3].plot(spectrum.x, stages["normalized"], lw=0.9, color="#9467bd")\n    axes[3].set_title("L2-normalized")\n    axes[-1].set_xlabel("Raman shift (cm$^{-1}$)")\n    for ax in axes:\n        ax.set_ylabel("Intensity")\n        ax.set_xlim(400, 3500)\n    fig.tight_layout()\n    fig.savefig(out_path, dpi=220, bbox_inches="tight")\n    plt.close(fig)\n\n\ndef peak_table(curves: dict[str, np.ndarray], out_path: Path) -> pd.DataFrame:\n    rows = []\n    for method, y in curves.items():\n        peaks, props = find_peaks(y, prominence=max(y.max() * 0.03, 1e-12), distance=8)\n        if len(peaks) == 0:\n            peaks = np.array([int(np.argmax(y))])\n            prominences = np.array([y.max()])\n        else:\n            prominences = props["prominences"]\n        order = np.argsort(prominences)[::-1][:15]\n        for idx in peaks[order]:\n            shift = float(COMMON_SHIFT[idx])\n            lit = nearest_literature(shift)\n            rows.append({"method": method, "peak_cm-1": shift, **lit})\n    df = pd.DataFrame(rows)\n    df.to_csv(out_path, index=False)\n    return df\n\n\ndef nearest_literature(shift: float) -> dict:\n    diffs = np.abs(LITERATURE_TABLE["band_cm-1"].to_numpy(float) - shift)\n    row = LITERATURE_TABLE.iloc[int(np.argmin(diffs))]\n    delta = float(diffs.min())\n    if delta <= 15:\n        confidence = "close match"\n    elif delta <= 40:\n        confidence = "approximate"\n    else:\n        confidence = "unassigned"\n    return {\n        "matched_band_cm-1": float(row["band_cm-1"]) if confidence != "unassigned" else np.nan,\n        "functional_group": row["functional_group"] if confidence != "unassigned" else "Unassigned",\n        "assignment": row["assignment"] if confidence != "unassigned" else "Unassigned",\n        "source_type": row["source_type"] if confidence != "unassigned" else "Unassigned",\n        "match_delta_cm-1": delta,\n        "match_confidence": confidence,\n    }\n\n\ndef distance_metrics(matrix: np.ndarray, consensus: np.ndarray, weights: np.ndarray) -> pd.DataFrame:\n    rows = []\n    fp_mask = (COMMON_SHIFT >= FINGERPRINT_BOUNDS[0]) & (COMMON_SHIFT <= FINGERPRINT_BOUNDS[1])\n    ch_mask = (COMMON_SHIFT >= CH_STRETCH_BOUNDS[0]) & (COMMON_SHIFT <= CH_STRETCH_BOUNDS[1])\n    cprob = np.clip(consensus, 0, None)\n    cprob = cprob / cprob.sum()\n    for row in matrix:\n        rprob = np.clip(row, 0, None)\n        rprob = rprob / rprob.sum()\n        rows.append(\n            {\n                "l2": float(np.linalg.norm(row - consensus)),\n                "weighted_l2": float(np.sqrt(np.sum(weights * (row - consensus) ** 2))),\n                "auc": float(abs(np.trapezoid(row, COMMON_SHIFT) - np.trapezoid(consensus, COMMON_SHIFT))),\n                "wasserstein_1d": float(wasserstein_distance(COMMON_SHIFT, COMMON_SHIFT, u_weights=rprob, v_weights=cprob)),\n                "fingerprint_distance": float(np.linalg.norm(row[fp_mask] - consensus[fp_mask])),\n                "ch_stretch_distance": float(np.linalg.norm(row[ch_mask] - consensus[ch_mask])),\n            }\n        )\n    return pd.DataFrame(rows)\n\n\ndef save_matrix_csv(path: Path, matrix: np.ndarray, sample_names: list[str]) -> None:\n    pd.DataFrame(matrix, index=pd.Index(sample_names, name="sample_id"), columns=[f"I_{x:.2f}" for x in COMMON_SHIFT]).to_csv(path)\n\n\ndef plot_consensus(curves: dict[str, np.ndarray], out_path: Path) -> None:\n    fig, ax = plt.subplots(figsize=(13, 6))\n    for name, y in curves.items():\n        ax.plot(COMMON_SHIFT, y, lw=1.4, label=name)\n    ax.set(title="Corrected FDA Consensus Spectra", xlabel="Raman shift (cm$^{-1}$)", ylabel="Intensity (a.u.)", xlim=(400, 3500))\n    ax.legend(ncol=2)\n    fig.tight_layout()\n    fig.savefig(out_path, dpi=220, bbox_inches="tight")\n    plt.close(fig)\n\n\ndef plot_variance(std: np.ndarray, out_path: Path) -> None:\n    fig, ax = plt.subplots(figsize=(13, 5))\n    ax.plot(COMMON_SHIFT, std, lw=1.2)\n    ax.axvspan(1800, 2400, color="#ff7f0e", alpha=0.12, label="High natural variance noted earlier")\n    ax.axvspan(2700, 3500, color="#d62728", alpha=0.10, label="High natural variance noted earlier")\n    ax.set(title="Corrected Control Pointwise Standard Deviation", xlabel="Raman shift (cm$^{-1}$)", ylabel="Std dev", xlim=(400, 3500))\n    ax.legend()\n    fig.tight_layout()\n    fig.savefig(out_path, dpi=220, bbox_inches="tight")\n    plt.close(fig)\n\n\ndef run_corrected_pipeline() -> dict:\n    ensure_dirs()\n    controls = load_tabular_controls()\n    if len(controls) != 45:\n        raise ValueError(f"Expected 45 controls after combining structured datasets; got {len(controls)}")\n    control_manifest, functional_matrix, control_stages, linear_matrix = process_spectra(controls)\n    names = [s.name for s in controls]\n    control_manifest.to_csv(CORRECTED_DIR / "control_source_manifest.csv", index=False)\n    save_matrix_csv(CORRECTED_DIR / "corrected_control_functional_matrix.csv", functional_matrix, names)\n\n    plot_preprocessing_diagnostic(controls[0], control_stages[controls[0].name], CORRECTED_DIR / "preprocessing_stage_diagnostic.png")\n\n    arithmetic_mean = linear_matrix.mean(axis=0)\n    spline_mean = functional_matrix.mean(axis=0)\n    robust_median = geometric_median(functional_matrix)\n\n    sinkhorn_rows = []\n    sinkhorn_curve = None\n    for reg in [1e-4, 1e-5, 1e-6]:\n        candidate, info = sinkhorn_barycenter_numpy(functional_matrix, COMMON_SHIFT, reg)\n        sinkhorn_rows.append(info)\n        if candidate is not None and info.get("stable") and not info.get("converged") is False:\n            sinkhorn_curve = candidate\n            break\n    sinkhorn_trials = pd.DataFrame(sinkhorn_rows)\n    sinkhorn_trials.to_csv(CORRECTED_DIR / "wasserstein_sinkhorn_regularization_trials.csv", index=False)\n\n    quantile_barycenter = exact_quantile_barycenter(functional_matrix)\n    wasserstein_for_peaks = sinkhorn_curve if sinkhorn_curve is not None else quantile_barycenter\n    wasserstein_method = "sinkhorn_low_reg" if sinkhorn_curve is not None else "exact_quantile_fallback"\n\n    consensus = {\n        "arithmetic_mean_discrete_interpolated": arithmetic_mean,\n        "spline_mean_functional": spline_mean,\n        "robust_median_functional": robust_median,\n        f"wasserstein_{wasserstein_method}": wasserstein_for_peaks,\n        "wasserstein_exact_quantile": quantile_barycenter,\n    }\n    consensus_df = pd.DataFrame({"raman_shift": COMMON_SHIFT, **consensus})\n    consensus_df.to_csv(CORRECTED_DIR / "corrected_consensus_spectra.csv", index=False)\n    plot_consensus(consensus, CORRECTED_DIR / "corrected_consensus_overlay.png")\n    peak_matches = peak_table(consensus, CORRECTED_DIR / "consensus_peak_matches.csv")\n\n    std = functional_matrix.std(axis=0, ddof=1)\n    eps = max(np.median(std) * 0.05, np.finfo(float).eps)\n    weights = 1.0 / (std + eps)\n    weights *= len(weights) / weights.sum()\n    pd.DataFrame({"raman_shift": COMMON_SHIFT, "pointwise_std": std, "variance_weight": weights}).to_csv(\n        CORRECTED_DIR / "variance_weights.csv", index=False\n    )\n    plot_variance(std, CORRECTED_DIR / "corrected_variance_profile.png")\n\n    distances = distance_metrics(functional_matrix, robust_median, weights)\n    distances.insert(0, "sample_id", names)\n    distances.to_csv(CORRECTED_DIR / "corrected_control_distances.csv", index=False)\n    thresholds = []\n    for metric in ["l2", "weighted_l2", "auc", "wasserstein_1d"]:\n        values = distances[metric].to_numpy(float)\n        thresholds.append(\n            {\n                "distance_metric": metric,\n                "mean": values.mean(),\n                "std": values.std(ddof=1),\n                "90th_percentile": np.percentile(values, 90),\n                "95th_percentile": np.percentile(values, 95),\n                "99th_percentile": np.percentile(values, 99),\n                "chosen_threshold": np.percentile(values, THRESHOLD_PERCENTILE),\n            }\n        )\n    thresholds_df = pd.DataFrame(thresholds)\n    thresholds_df.to_csv(CORRECTED_DIR / "corrected_distance_thresholds.csv", index=False)\n\n    # NMF component audit on corrected functional control matrix.\n    nmf_rows = []\n    model_rows = []\n    for k in range(3, 7):\n        model = NMF(n_components=k, init="nndsvda", solver="cd", max_iter=5000, tol=1e-5, random_state=RANDOM_STATE)\n        W = model.fit_transform(functional_matrix)\n        H = model.components_\n        fig, ax = plt.subplots(figsize=(13, 6))\n        matched_milk = set()\n        assigned_components = 0\n        for j, comp in enumerate(H, start=1):\n            scaled = comp / max(comp.max(), 1e-12)\n            ax.plot(COMMON_SHIFT, scaled, lw=1.0, label=f"C{j}")\n            peaks, props = find_peaks(scaled, prominence=0.04, distance=8)\n            if len(peaks) == 0:\n                peaks = np.array([int(np.argmax(scaled))])\n                prominences = np.array([scaled.max()])\n            else:\n                prominences = props["prominences"]\n            component_assigned = False\n            for idx in peaks[np.argsort(prominences)[::-1][:7]]:\n                lit = nearest_literature(float(COMMON_SHIFT[idx]))\n                if lit["match_confidence"] != "unassigned":\n                    component_assigned = True\n                    if lit["source_type"] == "milk":\n                        matched_milk.add(lit["matched_band_cm-1"])\n                nmf_rows.append({"n_components": k, "component": j, "peak_cm-1": float(COMMON_SHIFT[idx]), **lit})\n            assigned_components += int(component_assigned)\n        ax.set(title=f"Corrected NMF Basis Spectra: k={k}", xlabel="Raman shift (cm$^{-1}$)", ylabel="Scaled component intensity", xlim=(400, 3500))\n        ax.legend(ncol=2)\n        fig.tight_layout()\n        fig.savefig(CORRECTED_DIR / f"nmf_components_k{k}.png", dpi=220, bbox_inches="tight")\n        plt.close(fig)\n        model_rows.append(\n            {\n                "n_components": k,\n                "unique_milk_bands_matched": len(matched_milk),\n                "components_with_assignment": assigned_components,\n                "milk_band_coverage_per_component": len(matched_milk) / k,\n                "relative_reconstruction_error": float(np.linalg.norm(functional_matrix - W @ H) / np.linalg.norm(functional_matrix)),\n            }\n        )\n    nmf_all = pd.DataFrame(nmf_rows)\n    nmf_models = pd.DataFrame(model_rows).sort_values(\n        ["milk_band_coverage_per_component", "unique_milk_bands_matched", "relative_reconstruction_error"],\n        ascending=[False, False, True],\n    )\n    selected_k = int(nmf_models.iloc[0]["n_components"])\n    nmf_all.to_csv(CORRECTED_DIR / "nmf_peak_assignments_all.csv", index=False)\n    nmf_models.to_csv(CORRECTED_DIR / "nmf_model_comparison.csv", index=False)\n    nmf_all[nmf_all["n_components"] == selected_k].to_csv(CORRECTED_DIR / "nmf_selected_component_summary.csv", index=False)\n\n    # Wavelet decomposition of corrected robust median.\n    coeffs = pywt.wavedec(robust_median, "sym8", level=4, mode="symmetric")\n\n    def reconstruct(keep_index: int) -> np.ndarray:\n        retained = [np.zeros_like(c) for c in coeffs]\n        retained[keep_index] = coeffs[keep_index]\n        return pywt.waverec(retained, "sym8", mode="symmetric")[: len(robust_median)]\n\n    approximation = reconstruct(0)\n    details = {level: reconstruct(4 - level + 1) for level in range(1, 5)}\n    fig, axes = plt.subplots(5, 1, figsize=(14, 13), sharex=True)\n    axes[0].plot(COMMON_SHIFT, approximation, lw=1.0)\n    axes[0].set_title("Approximation A4")\n    peak_region = ((COMMON_SHIFT >= 800) & (COMMON_SHIFT <= 1800)) | ((COMMON_SHIFT >= 2800) & (COMMON_SHIFT <= 3000))\n    wavelet_rows = []\n    for level, ax in zip(range(1, 5), axes[1:]):\n        detail = details[level]\n        ax.plot(COMMON_SHIFT, detail, lw=0.8)\n        ax.set_title(f"Detail D{level}")\n        wavelet_rows.append(\n            {\n                "detail_level": level,\n                "peak_region_energy_ratio": float(np.sum(detail[peak_region] ** 2) / np.sum(detail**2)),\n                "total_energy": float(np.sum(detail**2)),\n            }\n        )\n    for ax in axes:\n        ax.set_xlim(400, 3500)\n        ax.set_ylabel("Amplitude")\n    axes[-1].set_xlabel("Raman shift (cm$^{-1}$)")\n    fig.tight_layout()\n    fig.savefig(CORRECTED_DIR / "wavelet_decomposition.png", dpi=220, bbox_inches="tight")\n    plt.close(fig)\n    wavelet_summary = pd.DataFrame(wavelet_rows).sort_values("peak_region_energy_ratio", ascending=False)\n    wavelet_summary.to_csv(CORRECTED_DIR / "wavelet_level_summary.csv", index=False)\n\n    return {\n        "controls": controls,\n        "control_matrix": functional_matrix,\n        "control_names": names,\n        "consensus": consensus,\n        "robust_median": robust_median,\n        "weights": weights,\n        "thresholds": thresholds_df,\n        "distances": distances,\n        "selected_nmf_k": selected_k,\n        "wasserstein_method": wasserstein_method,\n        "sinkhorn_trials": sinkhorn_trials,\n    }\n\n\ndef evaluate_validation_sets(context: dict) -> dict:\n    bacterial = load_bacterial_tabular()\n    raw_samples, missing_raw = load_uploaded_raw_samples()\n    all_tests = bacterial + raw_samples\n    test_manifest, test_matrix, test_stages, _ = process_spectra(all_tests)\n    test_manifest.to_csv(NOTEBOOK3_DIR / "test_sample_manifest.csv", index=False)\n    save_matrix_csv(NOTEBOOK3_DIR / "test_functional_matrix.csv", test_matrix, [s.name for s in all_tests])\n\n    if raw_samples:\n        plot_preprocessing_diagnostic(raw_samples[0], test_stages[raw_samples[0].name], NOTEBOOK3_DIR / "uploaded_raw_preprocessing_diagnostic.png")\n    if bacterial:\n        plot_preprocessing_diagnostic(bacterial[0], test_stages[bacterial[0].name], NOTEBOOK3_DIR / "bacterial_preprocessing_diagnostic.png")\n\n    robust_median = context["robust_median"]\n    weights = context["weights"]\n    thresholds = context["thresholds"].set_index("distance_metric")["chosen_threshold"].to_dict()\n    test_dist = distance_metrics(test_matrix, robust_median, weights)\n    test_dist.insert(0, "sample_id", [s.name for s in all_tests])\n    test_dist.insert(1, "sample_group", [s.group for s in all_tests])\n    for metric in ["l2", "weighted_l2", "auc", "wasserstein_1d"]:\n        test_dist[f"normalized_{metric}"] = test_dist[metric] / thresholds[metric]\n    norm_cols = [f"normalized_{m}" for m in ["l2", "weighted_l2", "auc", "wasserstein_1d"]]\n    test_dist["authenticity_score"] = test_dist[norm_cols].mean(axis=1)\n    test_dist["above_threshold"] = test_dist["authenticity_score"] > 1.0\n    test_dist.to_csv(NOTEBOOK3_DIR / "test_distance_metrics_and_scores.csv", index=False)\n\n    control_dist = context["distances"].copy()\n    control_dist["sample_group"] = "control"\n    for metric in ["l2", "weighted_l2", "auc", "wasserstein_1d"]:\n        control_dist[f"normalized_{metric}"] = control_dist[metric] / thresholds[metric]\n    control_dist["authenticity_score"] = control_dist[norm_cols].mean(axis=1)\n    control_dist["above_threshold"] = control_dist["authenticity_score"] > 1.0\n\n    combined_for_roc = pd.concat(\n        [\n            control_dist[["sample_id", "sample_group", "l2", "weighted_l2", "auc", "wasserstein_1d", "authenticity_score", "above_threshold"]],\n            test_dist[["sample_id", "sample_group", "l2", "weighted_l2", "auc", "wasserstein_1d", "authenticity_score", "above_threshold"]],\n        ],\n        ignore_index=True,\n    )\n    combined_for_roc.to_csv(NOTEBOOK3_DIR / "control_and_test_scores_for_roc.csv", index=False)\n\n    metric_auc_rows = []\n    roc_plot_data = {}\n    bacterial_mask = combined_for_roc["sample_group"].isin(["control", "bacterial"])\n    binary_df = combined_for_roc.loc[bacterial_mask].copy()\n    y_true = (binary_df["sample_group"] == "bacterial").astype(int).to_numpy()\n    for metric in ["authenticity_score", "l2", "weighted_l2", "auc", "wasserstein_1d"]:\n        scores = binary_df[metric].to_numpy(float)\n        metric_auc = roc_auc_score(y_true, scores)\n        fpr, tpr, _ = roc_curve(y_true, scores)\n        metric_auc_rows.append({"metric": metric, "auroc_bacterial_vs_control": metric_auc})\n        roc_plot_data[metric] = (fpr, tpr, metric_auc)\n    metric_aucs = pd.DataFrame(metric_auc_rows).sort_values("auroc_bacterial_vs_control", ascending=False)\n    metric_aucs.to_csv(NOTEBOOK3_DIR / "bacterial_vs_control_metric_aurocs.csv", index=False)\n\n    fig, ax = plt.subplots(figsize=(7, 6))\n    for metric, (fpr, tpr, metric_auc) in roc_plot_data.items():\n        ax.plot(fpr, tpr, lw=1.4, label=f"{metric} (AUC={metric_auc:.3f})")\n    ax.plot([0, 1], [0, 1], "k--", lw=1)\n    ax.set(title="ROC: Bacterial Samples vs Corrected Controls", xlabel="False positive rate", ylabel="True positive rate")\n    ax.legend(fontsize=8)\n    fig.tight_layout()\n    fig.savefig(NOTEBOOK3_DIR / "roc_curves_bacterial_vs_control.png", dpi=220, bbox_inches="tight")\n    plt.close(fig)\n\n    plot_df = test_dist.copy()\n    fig, ax = plt.subplots(figsize=(13, 5.5))\n    colors = plot_df["sample_group"].map({"bacterial": "#d62728", "raw_unadulterated": "#2ca02c"}).fillna("#1f77b4")\n    ax.bar(np.arange(len(plot_df)), plot_df["authenticity_score"], color=colors)\n    ax.axhline(1.0, color="black", ls="--", lw=1.3, label="score = 1")\n    ax.set_xticks(np.arange(len(plot_df)))\n    ax.set_xticklabels(plot_df["sample_id"], rotation=75, ha="right", fontsize=8)\n    ax.set_ylabel("Authenticity score")\n    ax.set_title("Authenticity Scores for Bacterial and Uploaded Raw Milk Samples")\n    ax.legend()\n    fig.tight_layout()\n    fig.savefig(NOTEBOOK3_DIR / "authenticity_score_bar_chart.png", dpi=220, bbox_inches="tight")\n    plt.close(fig)\n\n    summary_rows = []\n    for group, group_df in combined_for_roc.groupby("sample_group"):\n        row = {\n            "sample_group": group,\n            "n": len(group_df),\n            "mean_authenticity_score": group_df["authenticity_score"].mean(),\n            "percent_above_threshold": 100.0 * group_df["above_threshold"].mean(),\n            "auroc_combined_score": np.nan,\n            "best_individual_metric": "",\n            "best_individual_metric_auroc": np.nan,\n        }\n        if group == "bacterial":\n            row["auroc_combined_score"] = float(metric_aucs.loc[metric_aucs["metric"] == "authenticity_score", "auroc_bacterial_vs_control"].iloc[0])\n            best_individual = metric_aucs[metric_aucs["metric"] != "authenticity_score"].iloc[0]\n            row["best_individual_metric"] = best_individual["metric"]\n            row["best_individual_metric_auroc"] = float(best_individual["auroc_bacterial_vs_control"])\n        summary_rows.append(row)\n    validation_summary = pd.DataFrame(summary_rows)\n    validation_summary.to_csv(NOTEBOOK3_DIR / "validation_summary_table.csv", index=False)\n\n    pd.DataFrame({"missing_uploaded_raw_sample": missing_raw}).to_csv(NOTEBOOK3_DIR / "missing_uploaded_samples.csv", index=False)\n    return {\n        "bacterial_count": len(bacterial),\n        "raw_count": len(raw_samples),\n        "missing_raw": missing_raw,\n        "test_scores": test_dist,\n        "summary": validation_summary,\n        "metric_aucs": metric_aucs,\n    }\n\n\ndef write_notes(context: dict, validation: dict) -> None:\n    def as_markdown(df: pd.DataFrame) -> str:\n        text_df = df.copy()\n        for col in text_df.columns:\n            if pd.api.types.is_float_dtype(text_df[col]):\n                text_df[col] = text_df[col].map(lambda v: "" if pd.isna(v) else f"{v:.6g}")\n            else:\n                text_df[col] = text_df[col].fillna("").astype(str)\n        headers = list(text_df.columns)\n        lines = ["| " + " | ".join(headers) + " |", "| " + " | ".join(["---"] * len(headers)) + " |"]\n        for _, row in text_df.iterrows():\n            lines.append("| " + " | ".join(str(row[col]) for col in headers) + " |")\n        return "\\n".join(lines)\n\n    thresholds = as_markdown(context["thresholds"])\n    summary = as_markdown(validation["summary"])\n    aurocs = as_markdown(validation["metric_aucs"])\n    missing = ", ".join(validation["missing_raw"]) if validation["missing_raw"] else "None"\n    notes = f"""# Corrected FDA Raman Validation Notes\n\n## What Was Fixed\n\n- Preprocessing now starts from the existing structured tabular datasets:\n  `src/combined_raman_spectra.xlsx` plus the explicit non-bacterial D2 control\n  in `src/combined_raman_spectra (Experimental).xlsx`. The previous generated\n  pipeline mixed raw-file loading and normalization-first behavior; this pass\n  applies the documented sequence: Savitzky-Golay smoothing, ALS baseline\n  correction, negative clipping, and L2 normalization.\n- Each preprocessed spectrum is fitted with a cubic B-spline (`splrep`, `k=3`,\n  `s=1e-4`) before functional evaluation on the common 400-3500 cm-1 grid.\n  This is the FDA step: it treats spectra as functions and makes the spline\n  mean distinct from the discrete interpolated arithmetic mean.\n- The Wasserstein curve is reframed as a shared dominant-peak locator. Low-reg\n  Sinkhorn trials are recorded in\n  `corrected_pipeline/wasserstein_sinkhorn_regularization_trials.csv`; the\n  stable fallback used for peak identification is `{context[\'wasserstein_method\']}`.\n- Notebook 2-style NMF, wavelet, variance weights, and thresholds were rerun\n  against the corrected functional control matrix.\n- Notebook 3-style validation was run against the 9 bacterial spectra in the\n  structured experimental workbook and the uploaded raw unadulterated samples.\n\n## Region Definitions and Citations\n\n- Fingerprint region: this implementation uses 400-1800 cm-1 because the\n  reference peak table includes milk-relevant protein, lipid, and carbohydrate\n  bands throughout that span. Literature boundaries vary: biomedical Raman work\n  commonly uses 400-1800 cm-1, while other summaries use roughly 600-1800 or\n  800-1800 cm-1.\n- CH-stretch region: this implementation uses 2800-3000 cm-1, within the\n  broader high-wavenumber region often described as 2800-3800 cm-1. The upper\n  bound is narrowed to the instrument range and the supplied milk table, where\n  fatty-acid peaks at 2865 and 2902 cm-1 are central.\n- Sources consulted:\n  - https://pmc.ncbi.nlm.nih.gov/articles/PMC2715834/\n  - https://pmc.ncbi.nlm.nih.gov/articles/PMC10052158/\n  - https://physicsopenlab.org/2022/01/11/raman-spectroscopy-of-organic-and-inorganic-molecules/\n  - https://pubs.acs.org/doi/10.1021/acs.analchem.5c05031\n\n## Corrected Thresholds\n\n{thresholds}\n\n## Validation Summary\n\n{summary}\n\n## Bacterial-vs-Control AUROC by Metric\n\n{aurocs}\n\n## Uploaded Raw Sample Limitation\n\nThe user described six unadulterated raw milk samples, but only five files were\npresent in Downloads: Samples 2-6. Missing file(s): {missing}. Because the\nuploaded raw set is all labelled unadulterated and contains no adulterated\ncounter-class, AUROC is not computed for that set; raw scores are reported.\n"""\n    (OUT_ROOT / "VALIDATION_README.md").write_text(notes, encoding="utf-8")\n\n\ndef main() -> None:\n    warnings.filterwarnings("ignore", category=UserWarning)\n    context = run_corrected_pipeline()\n    validation = evaluate_validation_sets(context)\n    write_notes(context, validation)\n    print("Corrected pipeline outputs:", CORRECTED_DIR)\n    print("Notebook 3 validation outputs:", NOTEBOOK3_DIR)\n    print("Validation notes:", OUT_ROOT / "VALIDATION_README.md")\n    print("Thresholds:")\n    print(context["thresholds"].to_string(index=False))\n    print("Validation summary:")\n    print(validation["summary"].to_string(index=False))\n    print("Metric AUROCs:")\n    print(validation["metric_aucs"].to_string(index=False))\n    if validation["missing_raw"]:\n        print("Missing uploaded raw sample files:", validation["missing_raw"])\n\n\nif __name__ == "__main__":\n    main()\n'
helper_path = Path('validation_output/run_corrected_fda_and_validation.py')
helper_path.parent.mkdir(parents=True, exist_ok=True)
helper_path.write_text(HELPER_SOURCE, encoding='utf-8')
print(f'Wrote corrected helper: {helper_path}')


In [ ]:
# Import the corrected implementation shared by the three notebooks.
import importlib.util
from pathlib import Path

module_path = Path.cwd() / "validation_output" / "run_corrected_fda_and_validation.py"
if not module_path.exists():
    raise FileNotFoundError(
        f"Missing {module_path}. Run this notebook from the corrected branch/files that include "
        "validation_output/run_corrected_fda_and_validation.py."
    )

spec = importlib.util.spec_from_file_location("corrected_raman_validation", module_path)
pipeline = importlib.util.module_from_spec(spec)
spec.loader.exec_module(pipeline)
print("Loaded corrected FDA Raman pipeline.")

## 2. Run Corrected Consensus Pipeline

The arithmetic mean is computed on linearly interpolated normalized spectra. The FDA spline mean, robust functional median, and Wasserstein peak-locator are computed from B-spline evaluated spectra, so the arithmetic mean and spline mean are intentionally distinct.

In [ ]:
context = pipeline.run_corrected_pipeline()
print(f"Controls used: {len(context['controls'])}")
print(f"Selected NMF k downstream: {context['selected_nmf_k']}")
print(f"Wasserstein method used for peak location: {context['wasserstein_method']}")

## 3. Preprocessing Diagnostic

In [ ]:
from IPython.display import Image, display
from pathlib import Path
out = Path("validation_output/corrected_pipeline")
display(Image(filename=str(out / "preprocessing_stage_diagnostic.png")))

## 4. Consensus Curves and Peak Matches

In [ ]:
import pandas as pd
from IPython.display import display, Image
out = Path("validation_output/corrected_pipeline")
display(Image(filename=str(out / "corrected_consensus_overlay.png")))
display(pd.read_csv(out / "consensus_peak_matches.csv").head(25))

## 5. Saved Outputs

In [ ]:
for path in sorted((Path("validation_output/corrected_pipeline")).glob("corrected_*")):
    print(path)
print(Path("validation_output/corrected_pipeline/control_source_manifest.csv"))
print(Path("validation_output/corrected_pipeline/consensus_peak_matches.csv"))